# MODUL 1 — TEXT PROCESSING PIPELINE
### Tugas Akhir Modul 1 · Natural Language Processing

| | |
|---|---|
| **Nama** | Minazati Ifada |
| **NIM** | 240306035 |
| **Dataset** | Indonesian Economic News Sentiment Q1 2026 (Kaggle) |
| **Output** | `modul1_final.ipynb` · `output/dataset_preprocessed.csv` |

Notebook ini menggabungkan seluruh konsep Pertemuan 1–5 menjadi satu pipeline end-to-end:

```text
DATASET CSV → LOAD DATA → CORPUS EXPLORATION → RAW TEXT → SENTENCE SEGMENTATION → TOKENIZATION
→ NORMALIZATION → STOPWORD REMOVAL → STEMMING → LINGUISTIC ANALYSIS (POS, Parsing) → N-GRAM → LANGUAGE MODEL
```

**Catatan urutan.** *Stopword removal* dan *stemming* menghapus kata dan merusak bentuk kata, sehingga struktur kalimat hilang.
Karena itu **POS tagging dan dependency parsing dijalankan pada kalimat bersih yang masih utuh** (Bagian 9–10), sedangkan
**n-gram dan language model memakai teks hasil stemming** (Bagian 11–12). Keputusan ini dijelaskan lagi di bagian analisis.

**Daftar isi:** 1 Dataset Description · 2 Corpus Exploration · 3 Data Cleaning · 4 Sentence Segmentation · 5 Tokenization ·
6 Normalization · 7 Stopword Removal · 8 Stemming · 9 POS Tagging · 10 Dependency Parsing · 11 N-Gram · 12 Language Model ·
13 Visualization · 14 Analysis · 15 Conclusion

In [2]:
import os

print("Folder kerja saat ini:")
print(os.getcwd())

print("\nIsi folder saat ini:")
print(os.listdir("."))

Folder kerja saat ini:
c:\Users\Lenovo\Downloads

Isi folder saat ini:
['01. Surat Rekomendasi Tokoh Masyarakat_BAZNAS Prov_Kab_Kot-1.doc', '02. Surat Penyataan.docx', '03. B. Akad Peserta Beasiswa - Reguler (1).docx', '03. B. Akad Peserta Beasiswa - Reguler.docx', '1.jpeg', '1306-ArticleText-6376-1-10-20230522 (1).pdf', '1306-ArticleText-6376-1-10-20230522.pdf', '133-581-1-PB.pdf', '15. Buku PDB.pdf', '1515-3968-1-PB.pdf', '16730-Article Text-71793-1-10-20260127.pdf', '2.3.8', '2.3.8.zip', '2.jpeg', '240306035_Minazati Ifada.pdf', '2995-6782-1-SM(PD) (1).pdf', '2995-6782-1-SM(PD) (2).pdf', '2995-6782-1-SM(PD).pdf', '3.jpeg', '38833FF26BA1D.UnigramPreview_g9c9v27vpyspw!App', '4.jpeg', '6478-Article Text-16674-1-10-20250717.pdf', '964.docx', 'Ai', 'Ai (1).pdf', 'Ai.pdf', 'AKAD.pdf', 'AKTIF-KULIAH-1.rtf', 'aktivitas_lms.csv', 'Analisis Layar Individu.pdf', 'Antigravity-x64.exe', 'anum4scan.pdf', 'APAxtKzg2c5eNM25toobyh.jpg', 'archive', 'archive.zip', 'Artificial Intelligence - A Modern A

In [4]:
import os

os.chdir(r"C:\Users\Lenovo\Documents\NLP")

print("Folder kerja:", os.getcwd())

Folder kerja: C:\Users\Lenovo\Documents\NLP


In [7]:
# ===== SETUP =====
# Jika library belum terpasang, jalankan sekali (hapus tanda #):
# %pip install pandas numpy matplotlib nltk Sastrawi
# %pip install stanza        # opsional: untuk POS & parsing bahasa Indonesia (Bagian 9-10)

import os, re, math, time, random
from collections import Counter, defaultdict
from itertools import chain

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.util import bigrams, trigrams
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

for resource in ["punkt", "punkt_tab", "stopwords"]:
    nltk.download(resource, quiet=True)

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 100
RANDOM_STATE = 42

# Lokasi file mengikuti struktur folder pengumpulan (dataset/ dan output/)
# Jika dataset berada di tempat lain, isi jalurnya di sini, contoh: MANUAL_PATH = r"C:\Users\Lenovo\Documents\NLP\modul-1\dataset.csv"
MANUAL_PATH = r"C:\Users\Lenovo\Documents\NLP\dataset.csv"

def find_dataset():
    if MANUAL_PATH:
        return MANUAL_PATH if os.path.exists(MANUAL_PATH) else None
    base = os.path.abspath(os.getcwd())
    # cari di folder kerja dan sampai 3 folder induk di atasnya
    folders = [base] + [os.path.abspath(os.path.join(base, *[".."] * i)) for i in range(1, 4)]
    for folder in folders:
        for name in ["dataset/dataset.csv", "dataset.csv"]:
            p = os.path.join(folder, name)
            if os.path.exists(p):
                return p
    return None

DATA_PATH = find_dataset()
if DATA_PATH is None:
    print("Folder kerja saat ini:", os.getcwd())
    print("Isi folder             :", os.listdir("."))
    raise FileNotFoundError(
        "dataset.csv tidak ditemukan. Letakkan di folder dataset/ (sejajar dengan notebook), "
        "atau isi MANUAL_PATH dengan jalur lengkap file CSV.")
os.makedirs("output", exist_ok=True)
OUTPUT_PATH = "output/dataset_preprocessed.csv"

print("Dataset   :", DATA_PATH)
print("Output    :", OUTPUT_PATH)
print("Versi     : pandas", pd.__version__, "| nltk", nltk.__version__)

Folder kerja saat ini: C:\Users\Lenovo\Documents\NLP
Isi folder             : ['.git', 'Contoh1.ipynb', 'Dataset', 'Fact_Article_Sentiment_Q1.csv', 'modul-1']


FileNotFoundError: dataset.csv tidak ditemukan. Letakkan di folder dataset/ (sejajar dengan notebook), atau isi MANUAL_PATH dengan jalur lengkap file CSV.

---
## 1. Dataset Description

Dataset berisi artikel berita ekonomi dan pasar modal Indonesia dari tiga media (Kontan, CNBC Indonesia, Bisnis.com)
selama kuartal pertama 2026. Setiap artikel punya label sentimen (`POSITIF`, `NETRAL`, `NEGATIF`), skor polaritas, dan skor
`ai_confidence`. Nama kolom `ai_confidence` menunjukkan bahwa label kemungkinan besar dibuat otomatis oleh model AI, bukan
anotasi manual. Hal ini perlu dicek di halaman Kaggle dan diperlakukan hati-hati bila label dipakai sebagai *ground truth*.
Pada tugas ini label hanya dipakai untuk eksplorasi; n-gram dan language model tidak memakai label.

In [ ]:
raw = pd.read_csv(DATA_PATH)
TEXT_COL, LABEL_COL = "content_article", "sentiment_label"
PLACEHOLDER = "Format Non-Text/Gagal Ekstrak"

deskripsi = {
    "Nama Dataset": "Indonesian Economic News Sentiment Q1 2026",
    "Sumber"      : "Kaggle (pengunggah: yuttasihinggusti); artikel dari Kontan, CNBC Indonesia, Bisnis.com",
    "URL Kaggle"  : "https://www.kaggle.com/datasets/yuttasihinggusti/indonesian-economic-news-sentiment-q1-2026",
    "Jumlah Data" : f"{len(raw):,} baris (1 baris = 1 artikel)",
    "Jumlah Kolom": f"{raw.shape[1]} kolom: {', '.join(raw.columns)}",
    "Kolom Teks"  : f"{TEXT_COL} (isi artikel); title_article = judul artikel",
    "Kolom Label" : f"{LABEL_COL} ({', '.join(sorted(raw[LABEL_COL].unique()))}); pendukung: polarity_score, ai_confidence",
    "Bahasa"      : "Indonesia (ragam jurnalistik formal) dengan sisipan istilah asing (net sell, support, resistance, dll.)",
    "Jenis Data"  : f"Teks berita ekonomi/pasar modal, terbit {raw['date_publication'].min()} s.d. {raw['date_publication'].max()}",
}
for k, v in deskripsi.items():
    print(f"{k:<13}: {v}")

display(pd.DataFrame({"tipe data": raw.dtypes.astype(str), "contoh isi (60 karakter)": raw.iloc[0].astype(str).str[:60]}))

---
## 2. Corpus Exploration

Tahap ini memeriksa data **sebelum diubah apa pun**: bentuk data, nilai kosong, duplikat, distribusi label dan sumber,
panjang dokumen, serta pola teks yang perlu dibersihkan.

In [ ]:
print("Ukuran dataset:", raw.shape)
display(raw.head())
display(raw.tail(3))
raw.info()

In [ ]:
# Missing value dan duplikasi
print("Missing value per kolom:")
print(raw.isnull().sum().to_string())

n_placeholder = int((raw[TEXT_COL] == PLACEHOLDER).sum())
print(f"\nDokumen placeholder '{PLACEHOLDER}': {n_placeholder}")
print("Duplikat baris penuh  :", int(raw.duplicated().sum()))
print("Duplikat pada kolom teks:", int(raw.duplicated(subset=[TEXT_COL]).sum()))
print("Duplikat url_article  :", int(raw.duplicated(subset=["url_article"]).sum()))

In [ ]:
# Distribusi label, sumber media, dan waktu terbit
def dist_table(series):
    vc = series.value_counts()
    return pd.DataFrame({"jumlah": vc, "persen": (vc / vc.sum() * 100).round(2)})

print("Distribusi label sentimen:");  display(dist_table(raw[LABEL_COL]))
print("Distribusi sumber media:");    display(dist_table(raw["media_source"]))
print("Jumlah artikel per bulan:");   display(dist_table(raw["date_publication"].str[:7]).sort_index())

In [ ]:
# Panjang dokumen mentah (sebelum cleaning)
raw_chars = raw[TEXT_COL].astype(str).str.len()
raw_words = raw[TEXT_COL].astype(str).str.split().str.len()
display(pd.DataFrame({"karakter": raw_chars.describe(), "kata (split)": raw_words.describe()}).round(1))

print("Dokumen sangat pendek (< 50 kata):")
display(raw.loc[raw_words < 50, ["media_source", "title_article", TEXT_COL]])

In [ ]:
# Pola awal dan akhir artikel per media: apakah ada teks non-berita (boilerplate)?
valid = raw[raw[TEXT_COL] != PLACEHOLDER]
for src, grp in valid.groupby("media_source"):
    ex = grp[TEXT_COL].iloc[0]
    print(f"[{src}]")
    print(f"   AWAL : {ex[:170]!r}")
    print(f"   AKHIR: {ex[-140:]!r}\n")

print("Artikel mengandung 'Reporter:'  :", int(valid[TEXT_COL].str.contains("Reporter:", regex=False).sum()))
print("Artikel mengandung 'Editor:'    :", int(valid[TEXT_COL].str.contains("Editor:", regex=False).sum()))
print("Artikel mengandung 'Disclaimer' :", int(valid[TEXT_COL].str.contains("Disclaimer", regex=False).sum()))

**Temuan eksplorasi.**
1. Teks berita berisi **boilerplate** yang bukan isi berita: baris *byline* (`Reporter: ... | Editor: ...`), kepala berita
   (`KONTAN.CO.ID - JAKARTA.`, `Jakarta, CNBC Indonesia -`, `Bisnis.com, JAKARTA —`), dan *disclaimer* di akhir artikel.
   Jika dibiarkan, bagian ini akan mendominasi n-gram (misalnya trigram `kontan co id`) dan membuat language model belajar format
   penulisan media, bukan bahasa ekonomi.
2. Ada dokumen placeholder (`Format Non-Text/Gagal Ekstrak`) dan dokumen duplikat yang harus dibuang.
3. Label sangat tidak seimbang (didominasi `NETRAL`), sehingga label kurang cocok dijadikan dasar analisis tanpa penanganan khusus.

---
## 3. Data Cleaning

Pembersihan di tahap ini bersifat **struktural**: membuang dokumen yang tidak valid dan teks non-berita. Huruf besar-kecil, angka,
dan tanda baca **belum disentuh** karena masih dibutuhkan untuk segmentasi kalimat, tokenisasi, dan POS tagging.
Penghapusan huruf besar, angka, dan tanda baca dilakukan nanti pada tahap *normalization* (Bagian 6).

In [ ]:
data = raw.copy()
n_awal = len(data)

# 3.1 Buang nilai kosong dan placeholder gagal-ekstrak
data = data.dropna(subset=[TEXT_COL])
data = data[data[TEXT_COL] != PLACEHOLDER]
n_setelah_placeholder = len(data)

# 3.2 Buang duplikat berdasarkan isi artikel
data = data.drop_duplicates(subset=[TEXT_COL])
n_setelah_dup = len(data)

print(f"Jumlah dokumen awal              : {n_awal:,}")
print(f"Dibuang (kosong/placeholder)     : {n_awal - n_setelah_placeholder:,}")
print(f"Dibuang (duplikat isi artikel)   : {n_setelah_placeholder - n_setelah_dup:,}")
print(f"Jumlah dokumen setelah dibersihkan: {n_setelah_dup:,}")

In [ ]:
# 3.3 Hapus boilerplate (byline, kepala berita, disclaimer, tautan sumber)
# Nama editor Kontan diambil dari header yang lengkap. Daftar ini dipakai untuk membersihkan header yang tidak lengkap
# (tanpa "KONTAN.CO.ID") tanpa risiko memotong kalimat pertama artikel.
editors = (data[TEXT_COL].str.replace("\xa0", " ")
           .str.extract(r"\|\s*Editor\s*:\s*(.{2,60}?)\s*(?i:KONTAN\.CO\.ID)")[0].dropna().unique())
EDITOR_ALT = "|".join(sorted((re.escape(e) for e in editors), key=len, reverse=True))
print(f"Nama editor terdeteksi dari header lengkap: {len(editors)}")

HEADER_PATTERNS = [
    # Kontan: "Reporter: X | Editor: Y KONTAN.CO.ID - JAKARTA." (kota dan "KONTAN.CO.ID" boleh tidak ada)
    r"^\s*(?:Reporter|Sumber|Penulis)\s*:[^|]{0,80}\|\s*Editor\s*:\s*(?:" + EDITOR_ALT + r")\s*(?:(?i:KONTAN\.CO\.ID)\s*[-–—]\s*)?(?:[A-Z][A-Z ,/]{1,40}\.(?=\s|$))?",
    r"^\s*(?i:KONTAN\.CO\.ID)\s*[-–—]\s*(?:[A-Z][A-Z ,/]{1,40}\.(?=\s|$))?",
    # CNBC Indonesia: "Jakarta, CNBC Indonesia - "
    r"^\s*[A-Z][A-Za-z ]{2,20},\s*CNBC Indonesia\s*[-–—]?\s*",
    # Bisnis.com: "Bisnis.com, JAKARTA — " (kota bisa kapital semua atau huruf awal kapital)
    r"^\s*Bisnis\.com,\s*[A-Z][A-Za-z ]{2,25}\s*[-–—]\s*",
]
FOOTER_PATTERNS = [
    r"Disclaimer\s*:\s*Berita ini tidak bertujuan.*?keputusan investasi pembaca\.",
    r"Baca artikel CNBC Indonesia\s*\".*?\"\s*selengkapnya di sini\s*:?\s*https?://\S+",
    r"Download Apps CNBC Indonesia sekarang\s*https?://\S+",
    r"Sumber\s*:\s*https?://\S+",
]

def strip_boilerplate(text):
    text = str(text).replace("\xa0", " ")
    for pattern in HEADER_PATTERNS:
        text = re.sub(pattern, "", text, count=1, flags=re.S)
    for pattern in FOOTER_PATTERNS:
        text = re.sub(pattern, "", text, flags=re.S | re.I)
    return re.sub(r"\s+", " ", text).strip()

data["content_clean"] = data[TEXT_COL].apply(strip_boilerplate)

# Dokumen yang menjadi terlalu pendek setelah dibersihkan tidak informatif
too_short = data["content_clean"].str.split().str.len() < 10
print("Dokumen terlalu pendek (< 10 kata) yang dibuang:", int(too_short.sum()))
data = data[~too_short].reset_index(drop=True)

berubah = (data[TEXT_COL] != data["content_clean"]).sum()
print(f"Dokumen yang teksnya berubah: {berubah:,} dari {len(data):,}")
print(f"Karakter yang dibuang       : {(1 - data['content_clean'].str.len().sum() / data[TEXT_COL].str.len().sum()) * 100:.2f}%")

In [ ]:
# Bukti sebelum-sesudah (satu contoh per media)
for src in data["media_source"].unique():
    r = data[data["media_source"] == src].iloc[0]
    print(f"[{src}]")
    print(f"   SEBELUM: {r[TEXT_COL][:140]!r}")
    print(f"   SESUDAH: {r['content_clean'][:140]!r}\n")

# Cek sisa boilerplate di awal teks (penyebutan "Kontan.co.id" di dalam kutipan narasumber adalah isi berita, bukan boilerplate)
sisa = data["content_clean"].str.contains(
    r"^(?:Reporter|Sumber)\s*:|^(?i:KONTAN\.CO\.ID)|^[A-Za-z ]+, CNBC Indonesia|^Bisnis\.com,|Disclaimer\s*:", regex=True).sum()
print("Dokumen yang masih mengandung pola boilerplate:", int(sisa))

In [ ]:
# Perbandingan corpus sebelum dan sesudah cleaning
before = raw[TEXT_COL].astype(str)
after = data["content_clean"]
display(pd.DataFrame({
    "Sebelum cleaning": [len(before), before.str.len().sum(), before.str.split().str.len().sum()],
    "Sesudah cleaning": [len(after),  after.str.len().sum(),  after.str.split().str.len().sum()],
}, index=["Jumlah dokumen", "Jumlah karakter", "Jumlah kata (split)"]))

---
## 4. Sentence Segmentation

Memakai `sent_tokenize` dari NLTK. Punkt tidak punya model bahasa Indonesia, sehingga dipakai model `english`. Aturan titik-akhir-kalimat
pada kedua bahasa mirip, tetapi singkatan khas Indonesia (`Tbk.`, `Pjs.`, `dkk.`) tidak dikenal. Hasilnya perlu diperiksa, bukan dipercaya begitu saja.

In [ ]:
data["sentences"] = data["content_clean"].apply(lambda t: sent_tokenize(t, language="english"))
data["n_sentences"] = data["sentences"].apply(len)

print("Contoh segmentasi dokumen pertama:")
for i, s in enumerate(data.loc[0, "sentences"][:6], 1):
    print(f"  {i}. {s}")

print(f"\nTotal kalimat            : {data['n_sentences'].sum():,}")
print(f"Rata-rata kalimat/dokumen: {data['n_sentences'].mean():.1f}")
display(data["n_sentences"].describe().round(1).to_frame("kalimat per dokumen").T)

In [ ]:
# Error analysis segmentasi: kalimat yang berakhir dengan singkatan biasanya berarti pemotongan salah
all_sents = pd.Series(list(chain.from_iterable(data["sentences"])))
ABBR = r"\b(?:Tbk|Pjs|Rp|No|Dr|Ir|Prof|Hj|Bpk|dkk|dll|dsb|dst|yth)\.$"

suspect_abbr = all_sents[all_sents.str.contains(ABBR, regex=True)]
suspect_short = all_sents[all_sents.str.split().str.len() <= 2]
long_sents = all_sents[all_sents.str.split().str.len() > 80]

print(f"Total kalimat                          : {len(all_sents):,}")
print(f"Berakhir dengan singkatan (curiga salah): {len(suspect_abbr):,} ({len(suspect_abbr)/len(all_sents)*100:.2f}%)")
print(f"Sangat pendek (<= 2 kata)              : {len(suspect_short):,} ({len(suspect_short)/len(all_sents)*100:.2f}%)")
print(f"Sangat panjang (> 80 kata)             : {len(long_sents):,} ({len(long_sents)/len(all_sents)*100:.2f}%)")

print("\nContoh kalimat berakhir singkatan:")
for s in suspect_abbr.head(5): print("  -", s[-110:])
print("\nContoh kalimat sangat pendek:")
for s in suspect_short.head(8): print("  -", repr(s))
print("\nContoh kalimat sangat panjang (awal):")
for s in long_sents.head(2): print("  -", s[:200], "...")

**Analisis segmentasi.** Hasil di atas menunjukkan dua sumber kesalahan: (1) titik pada singkatan dianggap akhir kalimat, dan
(2) kalimat yang tidak diakhiri tanda baca, misalnya daftar rekomendasi saham atau kutipan, tidak terpotong sehingga menjadi sangat panjang.
Untuk berita ekonomi, kesalahan ini kecil secara proporsi, tetapi tetap berpengaruh pada statistik jumlah kalimat dan pada
POS/dependency parsing yang bekerja per kalimat.

---
## 5. Tokenization

Tokenisasi pada tiga level: **kata** (`word_tokenize`), **karakter** (`list`), dan perbandingan dengan `split()` serta tokenizer spaCy.
Hasil tokenizer diperiksa pada pola khas berita ekonomi: angka berformat Indonesia (`8.676,74`), mata uang (`Rp 2.837.000`), persen, tanggal,
singkatan, dan URL.

In [ ]:
# 5.1 Word tokenization pada seluruh corpus
t0 = time.time()
data["tokens_raw"] = data["content_clean"].apply(word_tokenize)
data["n_tokens_raw"] = data["tokens_raw"].apply(len)
data["n_words"] = data["content_clean"].str.split().str.len()
print(f"Tokenisasi {len(data):,} dokumen selesai dalam {time.time() - t0:.1f} detik")
print("Contoh token dokumen pertama:", data.loc[0, "tokens_raw"][:25])

display(data[["n_words", "n_tokens_raw"]].describe().round(1))
print(f"Rasio token/kata: {data['n_tokens_raw'].sum() / data['n_words'].sum():.3f}  (token > kata karena tanda baca dipisah)")

In [ ]:
# 5.2 Character tokenization
print("Character tokenization 'IHSG':", list("IHSG"))

sample_text = " ".join(data["content_clean"].sample(300, random_state=RANDOM_STATE))
char_counter = Counter(sample_text)
print(f"\nSampel 300 dokumen: {len(sample_text):,} karakter, {len(char_counter)} karakter unik")
print("15 karakter terbanyak:", [(repr(c), n) for c, n in char_counter.most_common(15)])
non_alnum = sorted((c for c in char_counter if not c.isalnum() and not c.isspace()), key=lambda c: -char_counter[c])
print("Simbol/tanda baca yang muncul:", "".join(non_alnum[:40]))

In [ ]:
# 5.3 Eksperimen tokenizer pada pola khas dataset
try:
    import spacy
    _nlp_blank = spacy.blank("id")          # tokenizer bahasa Indonesia, tanpa model statistik
    spacy_tok = lambda s: [t.text for t in _nlp_blank(s)]
except Exception as e:
    spacy_tok = None
    print("spaCy tidak tersedia, kolom spaCy dilewati:", type(e).__name__)

samples = [
    "IHSG ditutup menguat 0,84% ke level 8.676,74.",
    "Harga emas Antam Rp 2.837.000 per gram (28/3/2026).",
    "Harga emas spot US$ 4.322,61 per ons troi.",
    "PT Bank Rakyat Indonesia (Persero) Tbk. (BBRI) mencatat laba.",
    "Baca selengkapnya di https://www.kontan.co.id/news/emas",
    "Hubungi redaksi@kontan.co.id untuk informasi.",
    "S&P 500 dan Nasdaq melemah; Dow Jones stagnan.",
    "Pemerintah tidak menaikkan harga BBM.",
]
rows = []
for s in samples:
    row = {"Teks": s, "split()": s.split(), "NLTK word_tokenize": word_tokenize(s)}
    if spacy_tok:
        row["spaCy (id)"] = spacy_tok(s)
    rows.append(row)
pd.set_option("display.max_colwidth", 200)
display(pd.DataFrame(rows))
pd.set_option("display.max_colwidth", 110)

**Analisis tokenizer.** `split()` menempelkan tanda baca ke kata (`Tbk.`, `BBM.`), sedangkan `word_tokenize` memisahkannya tetapi tetap bisa
memecah entitas yang seharusnya utuh (URL, email, `S&P`). Tokenizer hanyalah alat; pada angka berformat Indonesia (titik sebagai pemisah ribuan,
koma sebagai desimal) hasilnya harus diperiksa. Untuk tujuan n-gram di notebook ini, angka dan tanda baca nantinya dibuang sehingga perbedaan
tersebut tidak berdampak besar, tetapi pada tugas lain (misalnya ekstraksi nilai transaksi) tokenizer khusus akan diperlukan.

In [ ]:
# 5.4 STATISTIK CORPUS (setelah cleaning)
idx_min, idx_max = data["n_words"].idxmin(), data["n_words"].idxmax()
stat_corpus = pd.Series({
    "Jumlah dokumen"         : f"{len(data):,}",
    "Jumlah kalimat"         : f"{data['n_sentences'].sum():,}",
    "Jumlah token"           : f"{data['n_tokens_raw'].sum():,}",
    "Jumlah kata (split)"    : f"{data['n_words'].sum():,}",
    "Jumlah karakter"        : f"{data['content_clean'].str.len().sum():,}",
    "Rata-rata kata/dokumen" : f"{data['n_words'].mean():.1f}",
    "Dokumen terpendek"      : f"{data.loc[idx_min, 'n_words']} kata (idx {idx_min}): {data.loc[idx_min, 'title_article'][:60]}",
    "Dokumen terpanjang"     : f"{data.loc[idx_max, 'n_words']} kata (idx {idx_max}): {data.loc[idx_max, 'title_article'][:60]}",
})
print("STATISTIK CORPUS")
print("=" * 70)
for k, v in stat_corpus.items():
    print(f"{k:<24}: {v}")

---
## 6. Normalization

Langkah normalisasi: *case folding* → hapus URL → hapus email → hapus mention → hapus hashtag → hapus angka → hapus tanda baca → rapikan spasi.
Tiap langkah dibuat sebagai fungsi terpisah (seperti Pertemuan 3) agar dampaknya bisa dianalisis satu per satu.

In [ ]:
def case_folding(text):      return str(text).lower()
def remove_url(text):        return re.sub(r"http\S+|www\.\S+", " ", text)
def remove_email(text):      return re.sub(r"\S+@\S+\.\S+", " ", text)
def remove_mention(text):    return re.sub(r"@\w+", " ", text)
def remove_hashtag(text):    return re.sub(r"#\w+", " ", text)
def remove_number(text):     return re.sub(r"\d+", " ", text)
def remove_punctuation(text):return re.sub(r"[^\w\s]", " ", text)
def remove_whitespace(text): return re.sub(r"\s+", " ", text).strip()

def clean_text(text):
    text = case_folding(text)
    text = remove_url(text)
    text = remove_email(text)
    text = remove_mention(text)
    text = remove_hashtag(text)
    text = remove_number(text)
    text = remove_punctuation(text)
    return remove_whitespace(text)

# Jejak transformasi pada satu kalimat dari corpus yang mengandung angka dan tanda baca
contoh = next((s for s in data.loc[0, "sentences"] if re.search(r"\d", s)), data.loc[0, "sentences"][0])
print("ASLI            :", contoh)
step = contoh
for fn in [case_folding, remove_url, remove_email, remove_mention, remove_hashtag, remove_number, remove_punctuation, remove_whitespace]:
    step = fn(step)
    print(f"{fn.__name__:<16}:", step)

In [ ]:
# Terapkan ke seluruh corpus lalu tokenisasi ulang
t0 = time.time()
data["clean_text"] = data["content_clean"].apply(clean_text)
data["tokens_norm"] = data["clean_text"].apply(word_tokenize)
print(f"Normalisasi selesai dalam {time.time() - t0:.1f} detik")

vocab_raw  = set(chain.from_iterable(data["tokens_raw"]))
vocab_norm = set(chain.from_iterable(data["tokens_norm"]))
print(f"Vocabulary token mentah        : {len(vocab_raw):,}")
print(f"Vocabulary setelah normalisasi : {len(vocab_norm):,}  (turun {(1 - len(vocab_norm)/len(vocab_raw))*100:.1f}%)")

In [ ]:
# Apakah semua tanda baca dan angka layak dihapus? Ukur seberapa banyak token yang terkena
tok_counter = Counter(chain.from_iterable(data["tokens_raw"]))
total_tok = sum(tok_counter.values())
punct_only = sum(c for t, c in tok_counter.items() if not any(ch.isalnum() for ch in t))
with_digit = sum(c for t, c in tok_counter.items() if re.search(r"\d", t))

print(f"Token berupa tanda baca saja : {punct_only:,} ({punct_only/total_tok*100:.1f}% dari semua token)")
print(f"Token mengandung angka       : {with_digit:,} ({with_digit/total_tok*100:.1f}% dari semua token)")
print("\n10 tanda baca terbanyak:", [(t, c) for t, c in tok_counter.most_common(60) if not any(ch.isalnum() for ch in t)][:10])
print("10 token berangka terbanyak:", [(t, c) for t, c in tok_counter.most_common(400) if re.search(r"\d", t)][:10])

# Contoh informasi penting yang hilang
ex = "IHSG ditutup naik 0,84% ke level 8.676,74 pada perdagangan Selasa (6/1/2026)."
print("\nAsli     :", ex)
print("Normalisasi:", clean_text(ex))

In [ ]:
# Apakah perlu normalisasi kata tidak baku (slang)? Cek dengan data, bukan asumsi
slang = ["yg", "gak", "nggak", "ga", "bgt", "udah", "aja", "dong", "sih", "nih", "banget", "lah", "kok", "deh"]
cnt_norm = Counter(chain.from_iterable(data["tokens_norm"]))
slang_found = {w: cnt_norm[w] for w in slang if cnt_norm[w] > 0}
total_norm = sum(cnt_norm.values())
print("Kata tidak baku yang ditemukan:", slang_found if slang_found else "tidak ada")
print(f"Total kemunculan: {sum(slang_found.values()):,} dari {total_norm:,} token ({sum(slang_found.values())/total_norm*100:.4f}%)")

**Analisis normalisasi.**
- **Tanda baca tidak selalu layak dihapus.** Pada berita ekonomi, tanda baca membawa makna: koma desimal (`0,84%`), titik ribuan (`2.837.000`), `%`,
  garis miring pada tanggal, dan titik akhir kalimat. Karena itu tanda baca **baru dihapus setelah segmentasi dan tokenisasi**, dan POS/parsing memakai
  teks yang tanda bacanya masih utuh.
- **Konsekuensi penghapusan angka.** Besaran perubahan harga atau nilai transaksi hilang, dan `Rp 17,43 triliun` menjadi `rp triliun`. Untuk
  language model berbasis urutan kata hal ini bisa diterima, tetapi untuk analisis yang butuh nilai angka, angka sebaiknya diganti token khusus (misalnya `<NUM>`), bukan dibuang.
- **Normalisasi slang tidak diperlukan.** Pemeriksaan di atas menunjukkan kata tidak baku nyaris tidak ada karena teks berasal dari media resmi.
  Hal ini berbeda dengan data media sosial yang menuntut kamus normalisasi.

---
## 7. Stopword Removal

Stopword tidak dihapus secara otomatis. Daftar stopword NLTK bahasa Indonesia dianalisis dulu: kata apa yang paling sering dibuang,
dan adakah kata yang **mengubah makna** jika dibuang. Kata negasi (`tidak`, `bukan`, `belum`, `tanpa`, ...) dipertahankan.

In [ ]:
stop_nltk = set(stopwords.words("indonesian"))
NEGATION = {"tidak", "bukan", "belum", "tanpa", "jangan", "tak"}

print(f"Jumlah stopword NLTK (Indonesia): {len(stop_nltk)}")
print("Kata negasi yang ada di daftar stopword NLTK :", sorted(NEGATION & stop_nltk))

# Daftar yang dipakai: stopword NLTK dikurangi kata negasi
stop_custom = stop_nltk - NEGATION

def remove_stopwords(tokens):
    # Buang stopword dan token 1 huruf (sisa tanda baca/singkatan seperti "s", "p" dari "S&P")
    return [t for t in tokens if t not in stop_custom and len(t) > 1]

cnt_before = Counter(chain.from_iterable(data["tokens_norm"]))
total_before = sum(cnt_before.values())
removed = Counter({w: c for w, c in cnt_before.items() if w in stop_custom or len(w) <= 1})
print(f"\nToken dibuang: {sum(removed.values()):,} dari {total_before:,} ({sum(removed.values())/total_before*100:.1f}%)")
print("15 stopword paling sering dibuang:", removed.most_common(15))

In [ ]:
# Contoh nyata: apa yang terjadi pada kalimat bernegasi jika "tidak" ikut dibuang?
neg_sent = next((s for s in chain.from_iterable(data["sentences"].head(500))
                 if re.search(r"\btidak\b", s.lower()) and 8 <= len(s.split()) <= 25),
                "Pemerintah tidak menaikkan harga BBM subsidi.")
toks = word_tokenize(clean_text(neg_sent))
print("Kalimat                      :", neg_sent)
print("Stopword NLTK penuh          :", [t for t in toks if t not in stop_nltk and len(t) > 1])
print("Negasi dipertahankan (dipakai):", remove_stopwords(toks))

In [ ]:
# Terapkan ke corpus dan periksa kata yang paling sering SETELAH stopword dibuang
data["tokens_nostop"] = data["tokens_norm"].apply(remove_stopwords)
cnt_after = Counter(chain.from_iterable(data["tokens_nostop"]))
print(f"Token sebelum : {total_before:,}")
print(f"Token sesudah : {sum(cnt_after.values()):,}  (turun {(1 - sum(cnt_after.values())/total_before)*100:.1f}%)")
print("\n20 kata terbanyak setelah stopword removal:")
print(cnt_after.most_common(20))

**Analisis stopword.** Stopword adalah kata fungsi (`yang`, `di`, `dan`, `untuk`) yang sangat sering muncul, sehingga menghapusnya
memperkecil corpus tanpa banyak kehilangan topik. Namun **tidak semua stopword aman dibuang**: tanpa `tidak`, kalimat
*"Pemerintah tidak menaikkan harga BBM"* menjadi *"pemerintah naik harga bbm"*, maknanya berbalik. Itu sebabnya negasi dipertahankan, apalagi dataset ini punya label sentimen.
Kata seperti `rp`, `pt`, dan `tbk` sangat sering muncul tetapi **tidak dibuang** karena merupakan bagian dari entitas (nama perusahaan, nilai uang) yang informatif untuk domain ekonomi.

---
## 8. Stemming

Memakai **Sastrawi** (algoritma Nazief–Adriani dengan kamus kata dasar). Stemming dijalankan **sekali per kata unik** lalu hasilnya dipetakan
kembali ke seluruh corpus (`stem_map`), jauh lebih cepat daripada menstem setiap kemunculan kata.

In [ ]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

vocab_nostop = set(chain.from_iterable(data["tokens_nostop"]))
t0 = time.time()
stem_map = {w: stemmer.stem(w) for w in vocab_nostop}
print(f"Stemming {len(vocab_nostop):,} kata unik selesai dalam {time.time() - t0:.1f} detik")

data["processed_tokens"] = data["tokens_nostop"].apply(lambda toks: [stem_map[t] for t in toks if stem_map[t]])
data["processed_text"] = data["processed_tokens"].apply(" ".join)

print("\nContoh stemming:")
for w in ["menguat", "penguatan", "melemah", "perdagangan", "keuangan", "investasi", "mencatatkan", "pertumbuhan", "kenaikan", "diperkirakan"]:
    if w in stem_map:
        print(f"  {w:<14} -> {stem_map[w]}")

In [ ]:
# Kata berimbuhan yang dikelompokkan ke satu stem (efek stemming pada vocabulary)
groups = defaultdict(set)
for w, s in stem_map.items():
    groups[s].add(w)

rows = []
for s, ws in sorted(groups.items(), key=lambda kv: -len(kv[1]))[:12]:
    top_forms = sorted(ws, key=lambda w: -cnt_after[w])[:6]
    rows.append({"stem": s, "jumlah bentuk kata": len(ws), "contoh bentuk (terbanyak)": ", ".join(top_forms)})
display(pd.DataFrame(rows))

print(f"Vocabulary sebelum stemming: {len(vocab_nostop):,}")
print(f"Vocabulary sesudah stemming: {len(groups):,}  (turun {(1 - len(groups)/len(vocab_nostop))*100:.1f}%)")

In [ ]:
# Error analysis stemming: (a) over-stemming, kata panjang menjadi sangat pendek; (b) kata berimbuhan yang tidak berubah
over = [(w, s) for w, s in stem_map.items() if w != s and len(s) <= 3 and len(w) >= 7]
over = sorted(over, key=lambda x: -cnt_after[x[0]])[:20]
print("(a) Curiga over-stemming (kata >= 7 huruf menjadi <= 3 huruf):")
print("    ", over)

under = [w for w, s in stem_map.items() if w == s and re.match(r"^(?:meng|mem|men|pen|per|ber|ter)\w{5,}", w)]
under = sorted(under, key=lambda w: -cnt_after[w])[:20]
print("\n(b) Kata berawalan imbuhan yang tidak berubah (mungkin sudah kata dasar, mungkin under-stemming):")
print("    ", under)

# Stemming mengubah kata dasar menjadi bentuk lain? bandingkan 'dagang' dan 'perdagangan'
print("\nPerlu dicek: ", {w: stem_map.get(w) for w in ["perdagangan", "pedagang", "dagang", "keuangan", "uang", "menguat", "kuat"]})

In [ ]:
# Fungsi pipeline lengkap (dipakai ulang untuk kalimat uji language model)
def preprocess_text(text):
    toks = word_tokenize(clean_text(text))
    toks = remove_stopwords(toks)
    return [stem_map[t] if t in stem_map else stemmer.stem(t) for t in toks if t]

# Tabel perbandingan tiap tahap pada beberapa kalimat nyata
sampel = data.sample(5, random_state=RANDOM_STATE)
rows = []
for _, r in sampel.iterrows():
    s = r["sentences"][min(2, len(r["sentences"]) - 1)]
    s = " ".join(s.split()[:22])
    t_clean = clean_text(s)
    t_tok = word_tokenize(t_clean)
    t_stop = remove_stopwords(t_tok)
    rows.append({"1 Raw": s, "2 Cleaning": t_clean, "3 Tokenisasi": t_tok, "4 Stopword": t_stop, "5 Stemming": preprocess_text(s)})
pd.set_option("display.max_colwidth", 160)
display(pd.DataFrame(rows))
pd.set_option("display.max_colwidth", 110)

In [ ]:
# Ringkasan efek tiap tahap preprocessing pada jumlah token dan ukuran vocabulary
stage_stats = pd.DataFrame({
    "Tahap": ["Token mentah (word_tokenize)", "Normalisasi", "Stopword removal", "Stemming"],
    "Jumlah token": [
        sum(len(t) for t in data["tokens_raw"]),
        sum(len(t) for t in data["tokens_norm"]),
        sum(len(t) for t in data["tokens_nostop"]),
        sum(len(t) for t in data["processed_tokens"]),
    ],
    "Vocabulary": [
        len(vocab_raw), len(vocab_norm), len(vocab_nostop),
        len(set(chain.from_iterable(data["processed_tokens"]))),
    ],
})
stage_stats["Vocab vs mentah (%)"] = (stage_stats["Vocabulary"] / stage_stats["Vocabulary"].iloc[0] * 100).round(1)
display(stage_stats)

# Simpan hasil preprocessing (dipakai Pertemuan 5 dan sebagai output tugas)
out_cols = ["date_publication", "media_source", "title_article", TEXT_COL, "content_clean", "clean_text",
            "processed_text", "sentiment_label", "polarity_score", "ai_confidence", "url_article"]
data[out_cols].to_csv(OUTPUT_PATH, index=False)
print(f"\nDisimpan: {OUTPUT_PATH}  ({len(data):,} dokumen)")

**Analisis stemming.** Stemming memperkecil vocabulary karena `menguat`, `penguatan`, dan `kuat` dikumpulkan ke satu bentuk, sehingga statistik n-gram lebih rapat
(mengurangi sparsity). Biayanya adalah hilangnya informasi morfologis dan munculnya kesalahan: kata yang berbeda makna bisa bertabrakan menjadi satu stem
(*over-stemming*), dan stem yang dihasilkan tidak selalu kata yang dipakai penulis (`perdagangan` → `dagang`, `keuangan` → `uang`).
Tabel di atas memperlihatkan contoh nyata dari dataset ini yang perlu dinilai satu per satu.

---
## 9. POS Tagging

POS tagging dijalankan pada **kalimat bersih yang masih utuh** (bukan hasil stopword/stemming), karena tagger membutuhkan urutan dan bentuk kata asli.

**Pilihan model.** spaCy tidak menyediakan model resmi bahasa Indonesia, dan model Inggris (`en_core_web_sm`) yang dipakai di Pertemuan 4 terbukti keliru pada teks Indonesia
(kata kerja `belajar` ditandai `ADJ`, seluruh kalimat informal menjadi `PROPN`). Maka notebook ini memakai **Stanza dengan model Indonesia (`id`)** sebagai pilihan utama.
Jika Stanza belum terpasang atau tidak ada internet untuk mengunduh model pertama kali, notebook otomatis memakai model Inggris spaCy sebagai *fallback*
dan menandainya jelas pada output.

In [ ]:
ENGINE, nlp_ling = None, None

try:
    import stanza
    nlp_ling = stanza.Pipeline("id", processors="tokenize,pos,lemma,depparse",
                               tokenize_no_ssplit=True, verbose=False, use_gpu=False)
    ENGINE = "stanza-id"
except Exception as e:
    print("Stanza (model Indonesia) tidak tersedia ->", type(e).__name__, ":", str(e)[:120])

if ENGINE is None:
    try:
        import spacy
        nlp_ling = spacy.load("en_core_web_sm")
        ENGINE = "spacy-en (FALLBACK: model bahasa Inggris, hasil pada teks Indonesia kurang akurat)"
    except Exception as e:
        print("spaCy en_core_web_sm juga tidak tersedia ->", type(e).__name__, ":", str(e)[:120])

print("\nEngine analisis linguistik:", ENGINE)


def parse_sentence(sentence):
    """Mengembalikan DataFrame: Token, POS, Lemma, Dependency, Head, HeadIdx (indeks 0-based, -1 = root)."""
    rows = []
    if ENGINE is None:
        return pd.DataFrame(columns=["Token", "POS", "Lemma", "Dependency", "Head", "HeadIdx"])
    if ENGINE.startswith("stanza"):
        doc = nlp_ling(sentence)
        offset = 0
        for sent in doc.sentences:
            for w in sent.words:
                head_idx = offset + w.head - 1 if w.head > 0 else -1
                head_txt = sent.words[w.head - 1].text if w.head > 0 else "ROOT"
                rows.append({"Token": w.text, "POS": w.upos, "Lemma": w.lemma, "Dependency": w.deprel,
                             "Head": head_txt, "HeadIdx": head_idx})
            offset += len(sent.words)
    else:
        doc = nlp_ling(sentence)
        for t in doc:
            rows.append({"Token": t.text, "POS": t.pos_, "Lemma": t.lemma_, "Dependency": t.dep_,
                         "Head": t.head.text if t.dep_ != "ROOT" else "ROOT",
                         "HeadIdx": t.head.i if t.dep_ != "ROOT" else -1})
    return pd.DataFrame(rows)

In [ ]:
# Ambil 100 kalimat acak (satu per dokumen, 8-22 token, tanpa URL) dari corpus
rng = random.Random(RANDOM_STATE)
sample_sents = []
for sents in data["sentences"].sample(frac=1, random_state=RANDOM_STATE):
    cands = [s for s in sents if 8 <= len(word_tokenize(s)) <= 22 and "http" not in s]
    if cands:
        sample_sents.append(rng.choice(cands))
    if len(sample_sents) == 100:
        break
print("Jumlah kalimat sampel:", len(sample_sents))

t0 = time.time()
parsed = [parse_sentence(s) for s in sample_sents]
print(f"Parsing {len(parsed)} kalimat selesai dalam {time.time() - t0:.1f} detik")

if ENGINE:
    for i in range(3):
        print(f"\nKalimat {i+1}: {sample_sents[i]}")
        display(parsed[i][["Token", "POS", "Lemma"]].T)

In [ ]:
# Distribusi POS pada 100 kalimat sampel
if ENGINE:
    pos_counts = Counter(chain.from_iterable(df["POS"] for df in parsed))
    pos_df = pd.DataFrame({"jumlah": pd.Series(pos_counts)}).sort_values("jumlah", ascending=False)
    pos_df["persen"] = (pos_df["jumlah"] / pos_df["jumlah"].sum() * 100).round(1)
    display(pos_df.T)
else:
    pos_counts = Counter()
    print("Engine tidak tersedia; bagian ini dilewati.")

**Analisis POS.** Pada berita ekonomi, komposisi POS didominasi `NOUN` dan `PROPN` (nama perusahaan, kode saham, institusi, indeks), diikuti `VERB`, `ADP`, dan `PUNCT`.
Kategori `PROPN` rawan salah: kode saham seperti `BBRI` atau istilah asing (`net sell`, `support`) sering tidak dikenali tagger. Manfaat POS di sini adalah
membedakan **entitas** (`PROPN`) dari **kata kerja pergerakan harga** (`menguat`, `melemah`, `ditutup`) yang menjadi inti kalimat berita ekonomi.

---
## 10. Dependency Parsing

Dependency parsing menentukan hubungan antar kata (siapa subjek, predikat, objek). Setiap kata memiliki satu *head* dan satu label relasi.
Hasil divisualisasikan dengan displaCy dan disimpan sebagai HTML di folder `output/`.

In [ ]:
if ENGINE:
    for i in range(3):
        print(f"\nKalimat {i+1}: {sample_sents[i]}")
        display(parsed[i][["Token", "POS", "Dependency", "Head"]])
else:
    print("Engine tidak tersedia; bagian ini dilewati.")

In [ ]:
# Distribusi relasi dependensi dan kategori POS dari kata ROOT (inti kalimat)
if ENGINE:
    dep_counts = Counter(chain.from_iterable(df["Dependency"].str.lower() for df in parsed))
    dep_df = pd.DataFrame({"jumlah": pd.Series(dep_counts)}).sort_values("jumlah", ascending=False)
    dep_df["persen"] = (dep_df["jumlah"] / dep_df["jumlah"].sum() * 100).round(1)
    print("10 relasi dependensi terbanyak:")
    display(dep_df.head(10).T)

    root_pos = Counter(df.loc[df["Dependency"].str.lower() == "root", "POS"].iloc[0]
                       for df in parsed if (df["Dependency"].str.lower() == "root").any())
    print("POS dari kata ROOT:", dict(root_pos.most_common()))
else:
    dep_counts = Counter()

In [ ]:
# Visualisasi pohon dependensi dengan displaCy (mode manual, bekerja untuk Stanza maupun spaCy)
def to_displacy(df):
    words = [{"text": r.Token, "tag": r.POS} for r in df.itertuples()]
    arcs = []
    for i, r in enumerate(df.itertuples()):
        h = r.HeadIdx
        if h < 0 or h == i:
            continue
        arcs.append({"start": min(i, h), "end": max(i, h), "label": r.Dependency, "dir": "left" if i < h else "right"})
    return {"words": words, "arcs": arcs}

if ENGINE:
    try:
        from spacy import displacy
        for i in range(2):
            data_dep = to_displacy(parsed[i])
            displacy.render(data_dep, style="dep", manual=True, jupyter=True, options={"distance": 90, "compact": True})
            html = displacy.render(data_dep, style="dep", manual=True, jupyter=False, page=True,
                                   options={"distance": 90, "compact": True})
            with open(f"output/dependency_tree_{i+1}.html", "w", encoding="utf-8") as f:
                f.write(html)
        print("Pohon dependensi disimpan di output/dependency_tree_1.html dan dependency_tree_2.html")
    except Exception as e:
        print("Visualisasi displaCy dilewati:", type(e).__name__, str(e)[:100])

In [ ]:
# Error analysis: tandai kalimat yang hasil parsingnya patut dicurigai secara otomatis, lalu periksa manual
def quick_checks(df):
    flags = []
    if df.empty:
        return ["kosong"]
    if "VERB" not in set(df["POS"]):
        flags.append("tanpa VERB")
    if (df["POS"] == "PROPN").mean() > 0.40:
        flags.append("PROPN > 40%")
    if {"X", "SYM"} & set(df["POS"]):
        flags.append("ada X/SYM")
    if (df["Dependency"].str.lower() == "root").sum() != 1:
        flags.append("ROOT != 1")
    return flags

if ENGINE:
    rows = []
    for s, df in zip(sample_sents[:10], parsed[:10]):
        rows.append({"Kalimat": s, "POS": " ".join(df["POS"]), "Peringatan otomatis": ", ".join(quick_checks(df)) or "-"})
    pd.set_option("display.max_colwidth", 170)
    display(pd.DataFrame(rows))
    pd.set_option("display.max_colwidth", 110)

    n_flag = sum(1 for df in parsed if quick_checks(df))
    print(f"Kalimat dengan minimal satu peringatan: {n_flag} dari {len(parsed)}")
    verbs = Counter(chain.from_iterable(df.loc[df["POS"] == "VERB", "Token"].str.lower() for df in parsed))
    print("Kata kerja terbanyak:", verbs.most_common(10))

**Analisis kesalahan linguistik.** Peringatan otomatis di atas hanya heuristik; penilaian akhir tetap manual dengan membandingkan label POS dan relasi terhadap kalimat.
Sumber kesalahan yang umum pada teks seperti ini: (1) kode saham dan nama perusahaan (`BBRI`, `PT ... Tbk`) ditebak sebagai kata biasa atau `X`;
(2) frasa nominal panjang (`Indeks Harga Saham Gabungan (IHSG)`) membuat *head* salah; (3) kalimat hasil segmentasi yang terpotong (Bagian 4)
menghasilkan struktur tidak lengkap; (4) bila *fallback* model Inggris yang aktif, kata kerja berimbuhan Indonesia (`menguat`, `ditutup`) sering salah label.
Mengamati pola kesalahan ini penting: **tagger dan parser adalah alat yang bisa salah**, sehingga hasilnya tidak boleh diterima begitu saja sebagai fakta linguistik.

---
## 11. N-Gram

N-gram dibangun dari `processed_tokens` (hasil normalisasi, stopword removal, dan stemming) **per dokumen**.
Berbeda dari Pertemuan 5 yang menggabungkan seluruh dokumen menjadi satu aliran token, di sini pasangan kata tidak melewati batas dokumen
sehingga tidak muncul bigram palsu antara kata terakhir artikel A dan kata pertama artikel B.

In [ ]:
docs = data["processed_tokens"].tolist()

unigram = Counter(chain.from_iterable(docs))
bigram = Counter(chain.from_iterable(bigrams(d) for d in docs))
trigram = Counter(chain.from_iterable(trigrams(d) for d in docs))

N_TOKENS = sum(unigram.values())
V = len(unigram)
print(f"Total token       : {N_TOKENS:,}")
print(f"Vocabulary (V)    : {V:,}")
print(f"Bigram unik       : {len(bigram):,}")
print(f"Trigram unik      : {len(trigram):,}")

def top_table(counter, n=20):
    total = sum(counter.values())
    rows = [{"Peringkat": i, "N-gram": " ".join(k) if isinstance(k, tuple) else k, "Frekuensi": c,
             "Proporsi (%)": round(c / total * 100, 3)} for i, (k, c) in enumerate(counter.most_common(n), 1)]
    return pd.DataFrame(rows).set_index("Peringkat")

print("\nTOP 20 UNIGRAM");  display(top_table(unigram))
print("TOP 20 BIGRAM");   display(top_table(bigram))
print("TOP 20 TRIGRAM");  display(top_table(trigram))

In [ ]:
# Sparsity: seberapa banyak ruang kombinasi yang benar-benar terisi?
def singleton_share(counter):
    return sum(1 for c in counter.values() if c == 1) / len(counter) * 100

sparsity = pd.DataFrame({
    "N-gram unik":            [len(unigram), len(bigram), len(trigram)],
    "Kemungkinan maksimum":   [V, V ** 2, V ** 3],
    "Terisi (%)":             [100.0, len(bigram) / V ** 2 * 100, len(trigram) / V ** 3 * 100],
    "Muncul hanya 1x (%)":    [singleton_share(unigram), singleton_share(bigram), singleton_share(trigram)],
}, index=["Unigram", "Bigram", "Trigram"])
pd.options.display.float_format = "{:,.6f}".format
display(sparsity)
pd.options.display.float_format = None

---
## 12. Language Model

Language model n-gram memperkirakan probabilitas kata atau kalimat dari frekuensi di corpus:

- Unigram: $P(w) = \dfrac{C(w)}{N}$
- Bigram (MLE): $P(w_i \mid w_{i-1}) = \dfrac{C(w_{i-1}, w_i)}{C(w_{i-1})}$
- Laplace (add-1): $P(w_i \mid w_{i-1}) = \dfrac{C(w_{i-1}, w_i) + 1}{C(w_{i-1}) + V}$
- Perplexity: $PP(W) = P(w_1 \ldots w_N)^{-1/N}$

### 12.1 Probabilitas unigram

In [ ]:
def p_unigram(w):
    return unigram[w] / N_TOKENS

uji_kata = ["saham", "harga", "pasar", "investor", "ihsg", "rupiah", "emas", "kucing"]
display(pd.DataFrame({
    "Frekuensi": [unigram[w] for w in uji_kata],
    "P(w)": [p_unigram(w) for w in uji_kata],
}, index=uji_kata))
print("'kucing' = out-of-vocabulary bila frekuensinya 0.")

### 12.2 Conditional probability bigram (MLE)

In [ ]:
def p_bigram(w1, w2):
    c1 = unigram[w1]
    return bigram[(w1, w2)] / c1 if c1 > 0 else 0.0

pairs = [("harga", "saham"), ("pasar", "modal"), ("suku", "bunga"), ("bursa", "efek"),
         ("nilai", "tukar"), ("saham", "harga"), ("belajar", "nlp")]
display(pd.DataFrame({
    "C(w1,w2)": [bigram[p] for p in pairs],
    "C(w1)": [unigram[p[0]] for p in pairs],
    "P(w2|w1)": [p_bigram(*p) for p in pairs],
}, index=[f"P({w2} | {w1})" for w1, w2 in pairs]))

### 12.3 Probabilitas kalimat dan perplexity (tanpa smoothing vs Laplace)

Kalimat uji ditulis dalam bentuk mentah lalu diproses dengan **pipeline yang sama** seperti corpus (`preprocess_text`), agar adil.
Tiga kalimat terakhir sengaja dibuat di luar domain atau diacak untuk memicu *zero probability*.

In [ ]:
def p_bigram_laplace(w1, w2):
    return (bigram[(w1, w2)] + 1) / (unigram[w1] + V)

def sentence_logprob(tokens, smoothed=False):
    """log P(kalimat) = jumlah log P(w_i | w_{i-1}). Mengembalikan -inf bila ada bigram berprobabilitas 0."""
    fn = p_bigram_laplace if smoothed else p_bigram
    logp = 0.0
    for a, b in zip(tokens, tokens[1:]):
        p = fn(a, b)
        if p == 0:
            return float("-inf")
        logp += math.log(p)
    return logp

def perplexity_from_logprob(logp, n):
    return float("inf") if logp == float("-inf") or n == 0 else math.exp(-logp / n)

test_raw = [
    "Indeks Harga Saham Gabungan ditutup menguat pada perdagangan hari ini.",
    "Bursa Efek Indonesia mencatat kenaikan nilai transaksi.",
    "Otoritas Jasa Keuangan menerbitkan aturan baru untuk pasar modal.",
    "Nilai tukar rupiah melemah terhadap dolar Amerika Serikat.",
    "Harga emas Antam naik di tengah ketidakpastian global.",
    "Dolar Amerika Serikat rupiah terhadap melemah nilai tukar.",      # diacak
    "Saya sedang belajar natural language processing.",                 # di luar domain
    "Kucing makan ikan di atas meja.",                                  # di luar domain
]

rows = []
for s in test_raw:
    toks = preprocess_text(s)
    n = max(len(toks) - 1, 0)                       # jumlah bigram
    lp_mle, lp_lap = sentence_logprob(toks, False), sentence_logprob(toks, True)
    rows.append({
        "Kalimat": s,
        "Token (setelah pipeline)": " ".join(toks),
        "N bigram": n,
        "P(W) MLE": math.exp(lp_mle) if lp_mle > float("-inf") else 0.0,
        "PP MLE": perplexity_from_logprob(lp_mle, n),
        "P(W) Laplace": math.exp(lp_lap) if lp_lap > float("-inf") else 0.0,
        "PP Laplace": perplexity_from_logprob(lp_lap, n),
    })
res = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 70)
pd.options.display.float_format = "{:,.4g}".format
display(res)
pd.options.display.float_format = None
pd.set_option("display.max_colwidth", 110)

### 12.4 Analisis zero probability

In [ ]:
print("Bigram penyebab probabilitas = 0 (tanpa smoothing):\n")
for s in test_raw:
    toks = preprocess_text(s)
    zero = [(a, b) for a, b in zip(toks, toks[1:]) if bigram[(a, b)] == 0]
    print(f"'{s}'  ->  {'ZERO PROBABILITY' if zero else 'aman'}")
    for a, b in zero:
        oov = [w for w in (a, b) if unigram[w] == 0]
        sebab = f"kata tidak ada di vocabulary: {oov}" if oov else "kedua kata ada, tetapi urutannya tidak pernah muncul"
        print(f"     ({a}, {b}) : {sebab}")

print(f"\nBigram yang pernah muncul    : {len(bigram):,} dari {V**2:,} kemungkinan ({len(bigram)/V**2*100:.4f}%)")
print(f"Bigram yang tidak pernah muncul: {100 - len(bigram)/V**2*100:.4f}%")

**Zero probability problem.** Probabilitas kalimat adalah hasil kali probabilitas bigram. Satu bigram yang tidak pernah muncul membuat seluruh kalimat berprobabilitas 0
dan perplexity tak hingga, termasuk untuk kalimat yang sebenarnya wajar. Penyebabnya ada dua: kata tidak ada di corpus (*out-of-vocabulary*), atau kedua kata ada tetapi
urutannya belum pernah teramati. Keduanya akibat *sparsity*: corpus hanya sampel kecil dari semua kemungkinan kalimat.
**Laplace smoothing** menambahkan 1 pada setiap hitungan dan $V$ pada penyebut sehingga tidak ada bigram yang bernilai 0.
Kelemahannya: karena $V$ besar, banyak massa probabilitas dialihkan ke bigram yang tidak pernah muncul, dan bigram yang sering muncul ikut turun probabilitasnya.

### 12.5 Prediksi kata berikutnya (bigram dan trigram)

In [ ]:
next_bi = defaultdict(list)
for (w1, w2), c in bigram.items():
    next_bi[w1].append((w2, c))
next_tri = defaultdict(list)
for (w1, w2, w3), c in trigram.items():
    next_tri[(w1, w2)].append((w3, c))

def predict_bigram(w, top_n=5):
    cands = sorted(next_bi.get(w, []), key=lambda x: -x[1])[:top_n]
    return [(nxt, (c + 1) / (unigram[w] + V)) for nxt, c in cands]           # probabilitas Laplace

def predict_trigram(w1, w2, top_n=5):
    cands = sorted(next_tri.get((w1, w2), []), key=lambda x: -x[1])[:top_n]
    return [(nxt, (c + 1) / (bigram[(w1, w2)] + V)) for nxt, c in cands]

def fmt(preds):
    return ", ".join(f"{w} ({p:.4f})" for w, p in preds) if preds else "(konteks tidak ada di corpus)"

print("PREDIKSI BIGRAM (1 kata konteks)")
for ctx in ["harga", "pasar", "bursa", "suku", "indeks", "nilai", "belajar"]:
    print(f"  {ctx:<10} -> {fmt(predict_bigram(ctx))}")

print("\nPREDIKSI TRIGRAM (2 kata konteks)")
for ctx in [("indeks", "harga"), ("bursa", "efek"), ("otoritas", "jasa"), ("nilai", "tukar"), ("suku", "bunga"), ("harga", "emas")]:
    print(f"  {' '.join(ctx):<16} -> {fmt(predict_trigram(*ctx))}")

### 12.6 Evaluasi pada data uji (held-out)

Perplexity pada kalimat yang **sudah ada di corpus** terlalu optimis karena model pernah melihatnya. Evaluasi yang adil memisahkan dokumen:
80% untuk melatih, 20% untuk menguji. Kata di data uji yang tidak ada di data latih diganti `<unk>`. Setiap dokumen diberi penanda awal `<s>` dan akhir `</s>`
agar unigram, bigram, dan trigram menilai jumlah prediksi yang sama sehingga perplexity-nya sebanding.
Diuji juga *add-k smoothing* (k kecil) sebagai pembanding Laplace (k = 1).

In [ ]:
idx = list(range(len(docs)))
random.Random(RANDOM_STATE).shuffle(idx)
cut = int(0.8 * len(idx))
train_docs = [docs[i] for i in idx[:cut]]
test_docs = [docs[i] for i in idx[cut:]]

vocab_train = set(chain.from_iterable(train_docs)) | {"</s>"}
V_train = len(vocab_train) + 1                       # +1 untuk <unk>

def build_counts(train, n):
    cnt, ctx = Counter(), Counter()
    for d in train:
        seq = ["<s>"] * (n - 1) + d + ["</s>"]
        for i in range(n - 1, len(seq)):
            g = tuple(seq[i - n + 1:i + 1])
            cnt[g] += 1
            ctx[g[:-1]] += 1
    return cnt, ctx

def heldout_perplexity(test, n, cnt, ctx, k):
    logp, m = 0.0, 0
    for d in test:
        d = [w if w in vocab_train else "<unk>" for w in d]
        seq = ["<s>"] * (n - 1) + d + ["</s>"]
        for i in range(n - 1, len(seq)):
            g = tuple(seq[i - n + 1:i + 1])
            logp += math.log((cnt.get(g, 0) + k) / (ctx.get(g[:-1], 0) + k * V_train))
            m += 1
    return math.exp(-logp / m)

n_test_tok = sum(len(d) for d in test_docs)
oov = sum(1 for d in test_docs for w in d if w not in vocab_train)
print(f"Dokumen latih/uji : {len(train_docs):,} / {len(test_docs):,}")
print(f"Token uji         : {n_test_tok:,}  | OOV rate: {oov / n_test_tok * 100:.2f}%  | V_train (+<unk>): {V_train:,}\n")

t0 = time.time()
hasil = {}
for n, nama in [(1, "Unigram"), (2, "Bigram"), (3, "Trigram")]:
    cnt, ctx = build_counts(train_docs, n)
    hasil[nama] = {f"add-{k}": heldout_perplexity(test_docs, n, cnt, ctx, k) for k in (1, 0.1, 0.01)}
print(f"Evaluasi selesai dalam {time.time() - t0:.1f} detik")

ppl_table = pd.DataFrame(hasil).T
pd.options.display.float_format = "{:,.1f}".format
display(ppl_table)
pd.options.display.float_format = None

best_model = ppl_table.stack().idxmin()
print(f"Perplexity terendah (terbaik): {best_model[0]} dengan {best_model[1]} -> {ppl_table.stack().min():,.1f}")

**Membaca tabel.** Perplexity yang lebih rendah berarti model lebih baik. Perhatikan dua hal: (1) pengaruh nilai *k*: Laplace murni (k = 1) memberi terlalu banyak massa
pada kombinasi yang tidak pernah muncul, dan pada model dengan konteks lebih panjang kerugian ini membesar; (2) efek panjang konteks: trigram punya konteks paling kaya tetapi paling sparse,
sehingga keunggulannya bisa hilang di data uji. Inilah *trade-off* utama n-gram.

---
## 13. Visualization

In [ ]:
# 1. Distribusi panjang dokumen
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(data["n_words"], bins=40, color="#4C78A8", edgecolor="white")
axes[0].axvline(data["n_words"].mean(), color="#E45756", ls="--", label=f"rata-rata = {data['n_words'].mean():.0f}")
axes[0].axvline(data["n_words"].median(), color="#54A24B", ls=":", label=f"median = {data['n_words'].median():.0f}")
axes[0].set_title("Distribusi panjang dokumen (kata)"); axes[0].set_xlabel("Jumlah kata"); axes[0].set_ylabel("Jumlah dokumen")
axes[0].legend()

order = ["Kontan", "CNBC", "Bisnis.com"]
axes[1].boxplot([data.loc[data["media_source"] == s, "n_words"] for s in order])
axes[1].set_xticklabels(order)
axes[1].set_title("Panjang dokumen per media"); axes[1].set_ylabel("Jumlah kata")
plt.tight_layout(); plt.show()

In [ ]:
# 2. Distribusi label sentimen
lab = data[LABEL_COL].value_counts()
fig, ax = plt.subplots(figsize=(6.5, 4.2))
bars = ax.bar(lab.index, lab.values, color=["#9D9D9D", "#E45756", "#54A24B"][:len(lab)])
for b, v in zip(bars, lab.values):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,}\n({v / lab.sum() * 100:.1f}%)", ha="center", va="bottom")
ax.set_title("Distribusi label sentimen (setelah cleaning)"); ax.set_ylabel("Jumlah dokumen")
ax.set_ylim(0, lab.max() * 1.18)
plt.tight_layout(); plt.show()

In [ ]:
# 3. Frekuensi top 20 unigram
words, counts = zip(*unigram.most_common(20))
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(words, counts, color="#4C78A8")
ax.set_title("20 kata paling sering muncul (unigram, setelah stemming)")
ax.set_ylabel("Frekuensi"); plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()

In [ ]:
# 4. Top 20 bigram dan trigram
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, counter, title, color in [(axes[0], bigram, "Top 20 bigram", "#F58518"), (axes[1], trigram, "Top 20 trigram", "#72B7B2")]:
    items = counter.most_common(20)[::-1]
    ax.barh([" ".join(k) for k, _ in items], [c for _, c in items], color=color)
    ax.set_title(title); ax.set_xlabel("Frekuensi")
plt.tight_layout(); plt.show()

In [ ]:
# 5. Ukuran vocabulary dan jumlah token pada tiap tahap preprocessing
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
short = ["Mentah", "Normalisasi", "Stopword", "Stemming"]
axes[0].bar(short, stage_stats["Jumlah token"], color="#4C78A8"); axes[0].set_title("Jumlah token per tahap")
axes[1].bar(short, stage_stats["Vocabulary"], color="#F58518");  axes[1].set_title("Ukuran vocabulary per tahap")
for ax in axes:
    ax.ticklabel_format(style="plain", axis="y")
    for lbl in ax.get_xticklabels(): lbl.set_rotation(15)
plt.tight_layout(); plt.show()

In [ ]:
# 6. Distribusi POS dan relasi dependensi (dari sampel kalimat)
if sum(pos_counts.values()) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    p = pd.Series(pos_counts).sort_values(ascending=False)
    axes[0].bar(p.index, p.values, color="#B279A2"); axes[0].set_title(f"Distribusi POS (100 kalimat; {ENGINE.split(' ')[0]})")
    d = pd.Series(dep_counts).sort_values(ascending=False).head(12)
    axes[1].bar(d.index, d.values, color="#FF9DA6"); axes[1].set_title("12 relasi dependensi terbanyak")
    for ax in axes: plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    plt.tight_layout(); plt.show()
else:
    print("Engine POS tidak tersedia; grafik POS dilewati.")

---
## 14. Analysis

### 14.1 Ringkasan angka dari seluruh pipeline

In [ ]:
bi_single = singleton_share(bigram)
tri_single = singleton_share(trigram)
n_dup_removed = n_setelah_placeholder - n_setelah_dup

display(Markdown(f"""
- **Dokumen:** {n_awal:,} awal → {len(data):,} setelah membuang {n_awal - n_setelah_placeholder} placeholder dan {n_dup_removed} duplikat.
- **Token:** {stage_stats['Jumlah token'].iloc[0]:,} (mentah) → {stage_stats['Jumlah token'].iloc[3]:,} (setelah stemming); **vocabulary** {stage_stats['Vocabulary'].iloc[0]:,} → {stage_stats['Vocabulary'].iloc[3]:,} (turun {100 - stage_stats['Vocab vs mentah (%)'].iloc[3]:.1f}%).
- **Sparsity:** {bi_single:.1f}% bigram dan {tri_single:.1f}% trigram hanya muncul satu kali; bigram mengisi {len(bigram)/V**2*100:.4f}% ruang kemungkinan, trigram {len(trigram)/V**3*100:.8f}%.
- **Perplexity data uji:** model terbaik adalah **{best_model[0]} ({best_model[1]})** dengan perplexity {ppl_table.stack().min():,.1f}.
- **Engine POS/parsing:** {ENGINE}.
"""))

### 14.2 Error analysis ringkas per tahap

| Tahap | Kesalahan yang teramati | Dampak | Penanganan |
|---|---|---|---|
| Cleaning | Boilerplate (byline, kepala berita, disclaimer) masuk ke teks | Mendominasi n-gram (`kontan co id`) | Dibuang dengan regex, lalu diverifikasi |
| Segmentasi | Singkatan (`Tbk.`) dan kalimat tanpa titik | Kalimat terpotong atau terlalu panjang | Diukur (Bagian 4); model Punkt bahasa Indonesia tidak tersedia |
| Tokenisasi | URL, email, `S&P` terpecah; angka format Indonesia perlu dicek | Entitas utuh rusak | Dibuang pada normalisasi karena tidak dibutuhkan n-gram |
| Normalisasi | Angka dan tanda baca dibuang | Nilai harga/persentase hilang | Dipertimbangkan token `<NUM>` untuk tugas yang butuh angka |
| Stopword | Negasi ikut dibuang bila memakai daftar NLTK penuh | Makna kalimat berbalik | Negasi dipertahankan |
| Stemming | Over-stemming dan stem bukan bentuk asli | Kata berbeda makna bertabrakan | Dianalisis di Bagian 8 |
| POS/Parsing | Kode saham dan istilah asing; frasa nominal panjang | Label dan *head* keliru | Heuristik peringatan + penilaian manual |
| N-gram | Sparsity, konteks pendek, tanpa makna | Probabilitas nol, prediksi dangkal | Smoothing, evaluasi held-out |

### 14.3 Mengapa tahap linguistic analysis tidak berada setelah stemming

Diagram pipeline di soal menempatkan *linguistic analysis* setelah stemming. Pada implementasi ini POS dan parsing dijalankan pada kalimat bersih yang belum di-stopword/stem,
karena (1) kata fungsi seperti `yang`, `di`, `untuk` dan imbuhan membawa informasi sintaksis yang dibutuhkan tagger dan parser, dan (2) hasil stemming (`menguat` → `kuat`)
bukan bentuk kata yang dikenal model. Kedua cabang (linguistik dan n-gram) sama-sama berasal dari teks yang sudah dibersihkan pada Bagian 3.

### 14.4 Jawaban Pertanyaan Refleksi Modul 1

**1. Mengapa komputer tidak dapat langsung memahami teks mentah?**
Komputer hanya mengolah angka. Teks mentah bagi komputer hanyalah deretan karakter tanpa batas kata, tanpa makna, dan tanpa hubungan antar kata.
Kalimat *"IHSG ditutup menguat 0,84%"* tidak memberi tahu mesin di mana kata berakhir, bahwa `menguat` dan `penguatan` berkaitan, atau bahwa angkanya adalah persentase.
Teks harus dipecah, dirapikan, lalu diubah ke representasi numerik (hitungan, probabilitas, dan nanti vektor) agar bisa dihitung.

**2. Apa perbedaan document, sentence, word, token, dan character?**
*Document* adalah satu satuan teks utuh (di dataset ini satu artikel berita). *Sentence* adalah satu kalimat di dalam dokumen. *Word* adalah kata yang dipisahkan spasi.
*Token* adalah satuan hasil tokenizer, bisa berupa kata, angka, atau tanda baca, sehingga jumlah token lebih banyak daripada jumlah kata (terlihat pada rasio token/kata di Bagian 5).
*Character* adalah satuan terkecil: huruf, angka, atau simbol.

**3. Mengapa tokenization merupakan tahap penting dalam NLP?**
Hampir semua tahap sesudahnya (stopword, stemming, n-gram, vektor) bekerja pada token. Jika tokenisasi salah, kesalahan menyebar ke seluruh pipeline dan menentukan isi vocabulary.
Contoh di dataset: `8.676,74` dan `kompas.com` mengandung titik dan koma yang bisa memecah satu entitas menjadi beberapa token.

**4. Apakah semua punctuation harus dihapus? Jelaskan berdasarkan dataset yang digunakan.**
Tidak. Pada berita ekonomi, tanda baca membawa makna: koma desimal (`0,84%`), titik ribuan (`2.837.000`), `%`, tanda kurung pada tanggal, dan titik akhir kalimat yang dibutuhkan untuk segmentasi dan parsing.
Karena itu tanda baca baru dihapus setelah segmentasi dan tokenisasi, dan POS/parsing memakai teks yang tanda bacanya masih utuh. Penghapusannya untuk n-gram punya harga: angka dan persentase ikut hilang.

**5. Apakah semua stopword harus dihapus? Berikan contoh ketika penghapusan stopword mengubah makna.**
Tidak. Kata negasi seperti `tidak`, `bukan`, `belum`, dan `tanpa` termasuk stopword pada beberapa daftar, tetapi membuangnya membalik makna.
*"Pemerintah tidak menaikkan harga BBM"* menjadi *"pemerintah naik harga bbm"*. Karena itu negasi dipertahankan.

**6. Apa perbedaan stemming dan lemmatization?**
Stemming memotong imbuhan dengan aturan sehingga hasilnya bisa bukan kata baku atau salah (over-stemming), dan tidak memperhatikan konteks. Lemmatization mengembalikan kata ke bentuk dasar kamus (lemma)
dengan memanfaatkan kamus, morfologi, dan sering informasi POS, sehingga lebih akurat tetapi lebih berat. Sastrawi berada di antara keduanya: berbasis aturan, tetapi memakai kamus kata dasar.

**7. Apa hubungan morphology dengan stemming?**
Morfologi mempelajari pembentukan kata. Bahasa Indonesia banyak memakai imbuhan (`me-`, `pe-`, `ber-`, `-an`, `-kan`, `per-an`, `ke-an`), misalnya `perdagangan` = `per-` + `dagang` + `-an`.
Stemming adalah proses membalik pembentukan itu dengan aturan morfologi untuk menemukan kata dasarnya.

**8. Apa manfaat POS Tagging dalam NLP?**
POS tagging memberi tahu kelas kata (kata benda, kerja, sifat, nama diri). Informasi ini membantu disambiguasi, pengenalan entitas, ekstraksi frasa, lemmatization, dan menjadi masukan parsing.
Pada berita ekonomi, POS membedakan entitas (`PROPN`) dari kata kerja pergerakan harga (`menguat`, `melemah`).

**9. Apa manfaat dependency parsing?**
Dependency parsing menunjukkan hubungan tata bahasa antar kata (siapa melakukan apa pada siapa). Hasilnya dipakai untuk ekstraksi relasi (*"IHSG ditutup menguat"*),
menentukan cakupan negasi, *question answering*, dan terjemahan mesin.

**10. Mengapa n-gram mengalami masalah sparsity?**
Jumlah kemungkinan kombinasi tumbuh sebesar $V^n$, sedangkan corpus terbatas. Sebagian besar kombinasi tidak pernah muncul, dan banyak yang hanya muncul sekali (lihat tabel sparsity di Bagian 11).
Akibatnya kombinasi yang sebenarnya wajar mendapat probabilitas nol pada data baru.

**11. Apa perbedaan unigram, bigram, dan trigram?**
Unigram menghitung kata secara independen tanpa konteks. Bigram memperhitungkan satu kata sebelumnya, trigram dua kata sebelumnya.
Konteks yang lebih panjang membuat prediksi lebih tajam (`suku` → `bunga`), tetapi datanya makin sparse sehingga butuh corpus jauh lebih besar atau smoothing.

**12. Mengapa neural language model diperlukan setelah n-gram?**
N-gram tidak menggeneralisasi: `mobil` dan `kendaraan` dianggap tidak berhubungan, konteksnya terbatas, tabelnya membengkak, dan kombinasi yang belum pernah dilihat sulit ditangani.
Neural language model memakai *embedding* sehingga kata yang mirip punya representasi yang mirip, dapat melihat konteks lebih panjang (RNN/LSTM/Transformer), dan berbagi parameter sehingga kombinasi baru tetap mendapat probabilitas yang masuk akal.

> *Catatan: jawaban di atas adalah draf. Tulis ulang dengan bahasa dan contoh sendiri sebelum dikumpulkan.*

---
## 15. Conclusion

In [ ]:
display(Markdown(f"""
1. **Teks mentah tidak bisa langsung dimodelkan.** Dataset berita ekonomi ini harus dibersihkan dari boilerplate ({n_awal - len(data)} dokumen dibuang, sisanya dirapikan)
   sebelum n-gram bermakna. Tanpa langkah itu, n-gram teratas hanyalah format penulisan media.
2. **Setiap tahap preprocessing adalah keputusan, bukan aturan mutlak.** Tanda baca baru dihapus setelah segmentasi, negasi dipertahankan, dan angka dibuang dengan sadar akan konsekuensinya.
   Ukuran vocabulary turun dari {stage_stats['Vocabulary'].iloc[0]:,} menjadi {stage_stats['Vocabulary'].iloc[3]:,} setelah seluruh tahap.
3. **Alat bisa salah.** Punkt (segmentasi), `word_tokenize`, Sastrawi, dan tagger/parser semuanya menghasilkan kesalahan yang terukur pada teks ini.
   Hasilnya harus diperiksa, terutama pada singkatan, angka berformat Indonesia, kode saham, dan istilah asing.
4. **Language model n-gram sederhana tetapi terbatas.** Ia mampu menghitung probabilitas dan memprediksi kata berikutnya dengan wajar (`suku` → `bunga`), namun menderita sparsity
   ({bi_single:.1f}% bigram dan {tri_single:.1f}% trigram hanya muncul sekali), memberi probabilitas nol pada kombinasi baru, tidak memahami makna, dan konteksnya pendek.
5. **Arah selanjutnya.** Keterbatasan tersebut menjadi motivasi Modul 2: representasi kata sebagai vektor (TF-IDF, *word embedding*) dan model neural (RNN, LSTM, Attention)
   yang menggeneralisasi antar kata dan melihat konteks lebih panjang.
"""))

### Checklist Mahasiswa

| | Item | Bagian |
|---|---|---|
| ✅ | Dataset berhasil dibaca | 1 |
| ✅ | Corpus dieksplorasi | 2 |
| ✅ | Missing value dan duplicate diperiksa | 2, 3 |
| ✅ | Jumlah dokumen, kata, dan kalimat dihitung | 3, 4, 5 |
| ✅ | Sentence segmentation, word tokenization, character tokenization | 4, 5 |
| ✅ | Case folding dan cleaning | 3, 6 |
| ✅ | Stopword dianalisis; stemming dilakukan | 7, 8 |
| ✅ | POS tagging dan dependency parsing | 9, 10 |
| ✅ | Unigram, bigram, trigram dibuat | 11 |
| ✅ | Probabilitas, smoothing, dan perplexity dihitung | 12 |
| ✅ | Visualisasi dibuat (6 grafik) | 13 |
| ✅ | Error analysis dan kesimpulan | 4–10, 14, 15 |